## Modelo 6 - BERT + Neural Network

La idea es usar todas las variables que veniamos utilizando antes pero también el titulo + descripción. Para transformar esto a variables numericas vamos a utilizar un sentence transformer para obtener los embeddings. Luego la cabeza de regresión del modelo será un red neuronal.

### Leemos los datos

In [1]:
import pandas as pd

train = pd.read_csv("datos_entrenamiento.csv")
valid = pd.read_csv("datos_validacion.csv")

### Construimos el texto

Generamos un campo donde tenemos lo siguiente:

Title: ....

Description: ...


In [2]:
train["texto"] = (
    "Title: " +
    train["title"].fillna("") +
    ". Description: " +
    train["description"].fillna("")
)

valid["texto"] = (
    "Title: " +
    valid["title"].fillna("") +
    ". Description: " +
    valid["description"].fillna("")
)

In [3]:
# import sys
# !{sys.executable} -m pip install sentence-transformers

### Obtenemos los embeddings

In [4]:
from sentence_transformers import SentenceTransformer

modelo = SentenceTransformer("all-MiniLM-L6-v2")

X_text_train = modelo.encode(
    train["texto"].tolist(),
    show_progress_bar=True
)

X_text_valid = modelo.encode(
    valid["texto"].tolist(),
    show_progress_bar=True
)

c:\Users\Sebastian\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Batches: 100%|██████████| 23/23 [00:11<00:00,  2.07it/s]


### Preparamos las otras variables

In [5]:
import numpy as np

variables_numericas = [
    "release_year",
    "runtime",
    "seasons",
    "log_imdb_votes",
    "votos_faltantes",
    "cantidad_generos",
    "cantidad_actores"
]

variables_categoricas = [
    "type",
    "age_certification",
    "genero_principal",
    "pais",
    "director"
]   

Pasamos las variables categoricas a numericas con OneHotEncoder

In [6]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

X_cat_train = encoder.fit_transform(
    train[variables_categoricas]
)

X_cat_valid = encoder.transform(
    valid[variables_categoricas]
)

Normalizamos las variables númericas

In [7]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_num_train = scaler.fit_transform(
    train[variables_numericas]
)

X_num_valid = scaler.transform(
    valid[variables_numericas]
)

Reducimos la dimensionalidad de los embeddings de la descripción + el título

In [8]:
from sklearn.decomposition import PCA

pca = PCA(
    n_components=100,
    random_state=123
)

X_text_train_pca = pca.fit_transform(X_text_train)

X_text_valid_pca = pca.transform(X_text_valid)

print(X_text_train.shape)
print(X_text_train_pca.shape)

print(
    "Varianza explicada:",
    pca.explained_variance_ratio_.sum()
)

(2907, 384)
(2907, 100)
Varianza explicada: 0.7189864


Concatenamos todas las variables: Text, Numericas y Categóricas

In [9]:
X_train = np.concatenate(
    [
        X_text_train,
        X_num_train,
        X_cat_train
    ],
    axis=1
)

X_valid = np.concatenate(
    [
        X_text_valid,
        X_num_valid,
        X_cat_valid
    ],
    axis=1
)

In [10]:
print(X_train.shape)
print(X_valid.shape)

(2907, 2203)
(724, 2203)


### Variable Respuesta

In [11]:
y_train = train["imdb_score"].values
y_valid = valid["imdb_score"].values

### Pasamos a tensores

Este paso es para poder implementar la red neuronal

In [12]:
import torch

X_train = torch.tensor(
    X_train,
    dtype=torch.float32
)

X_valid = torch.tensor(
    X_valid,
    dtype=torch.float32
)

y_train = torch.tensor(
    y_train,
    dtype=torch.float32
).reshape(-1,1)

y_valid = torch.tensor(
    y_valid,
    dtype=torch.float32
).reshape(-1,1)

### Implementación de la Red Neuronal

In [13]:
import torch
import torch.nn as nn

class MLPRegresion(nn.Module):

    def __init__(self, input_dim):

        super().__init__()

        self.modelo = nn.Sequential(

            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.20),

            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.20),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, 1)
        )

    def forward(self, x):

        return self.modelo(x)

Creamos la red

In [14]:
input_dim = X_train.shape[1]

modelo = MLPRegresion(input_dim)

### Función de pérdida y Optimizador

Como estamos usando MSE implementamos eso. Y usamos Adam como optimizador

In [15]:
criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    modelo.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

### Dataloader

Esto CREO que es para entrenar por batches en vez de usar cada fila para realizar una actualizacion de los pesos

In [16]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(
    X_train,
    y_train
)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

### Entrenamiento

In [17]:
import copy

num_epochs = 1500
patience = 30

best_val_loss = float("inf")
best_weights = None

train_losses = []
val_losses = []

epochs_without_improvement = 0

for epoch in range(num_epochs):

    # ==========================
    # Entrenamiento
    # ==========================

    modelo.train()

    train_loss = 0

    for X_batch, y_batch in train_loader:

        optimizer.zero_grad()

        pred = modelo(X_batch)

        loss = criterion(pred, y_batch)

        loss.backward()

        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)
    train_losses.append(train_loss)

    # ==========================
    # Validación
    # ==========================

    modelo.eval()

    with torch.no_grad():

        pred = modelo(X_valid)

        val_loss = criterion(
            pred,
            y_valid
        ).item()

    val_losses.append(val_loss)

    # ==========================
    # Early stopping
    # ==========================

    if val_loss < best_val_loss - 1e-4:

        best_val_loss = val_loss

        best_weights = copy.deepcopy(
            modelo.state_dict()
        )

        epochs_without_improvement = 0

    else:

        epochs_without_improvement += 1

    if (epoch + 1) % 10 == 0:

        print(
            f"Epoch {epoch+1:4d} | "
            f"Train: {train_loss:.4f} | "
            f"Valid: {val_loss:.4f}"
        )

    if epochs_without_improvement >= patience:

        print(
            f"\nEarly stopping en la época {epoch+1}."
        )

        break

# Recuperamos los mejores pesos

modelo.load_state_dict(best_weights)

print(
    f"\nMejor MSE de validación: {best_val_loss:.4f}"
)

Epoch   10 | Train: 0.5201 | Valid: 0.9067
Epoch   20 | Train: 0.3723 | Valid: 0.8643
Epoch   30 | Train: 0.3174 | Valid: 0.8481
Epoch   40 | Train: 0.2849 | Valid: 0.8937
Epoch   50 | Train: 0.2688 | Valid: 0.8542

Early stopping en la época 52.

Mejor MSE de validación: 0.8321


### Predicciones

In [18]:
modelo.eval()

with torch.no_grad():

    predicciones = modelo(X_valid)

mse = torch.mean(
    (predicciones - y_valid) ** 2
)

print("MSE:", mse.item())

MSE: 0.8320709466934204
